### 1. Basic Tasks

In [0]:
-- 1. Create a table, make 3 changes to it (insert, update, insert), and use DESCRIBE HISTORY to review the resulting versions.
create table if not exists cyntexa_dev.bronze.products (product_id int, product_name string, product_category string, price double)

In [0]:
insert into table cyntexa_dev.bronze.products values (101, 'Laptop', 'Electronics', 75000.00),
    (102, 'Office Chair', 'Furniture', 8500.00),
    (103, 'Wireless Mouse', 'Electronics', 1200.00),
    (104, 'Notebook', 'Stationery', 150.00),
    (105, 'Desk Lamp', 'Home & Office', 1800.00);

In [0]:
update cyntexa_dev.bronze.products 
set price = 130.00
where product_id = 104;

In [0]:
insert into cyntexa_dev.bronze.products values (106, 'Tablet', 'Electronics', 35000.00)

In [0]:
desc history cyntexa_dev.bronze.products

In [0]:
-- 2. Use COPY INTO to incrementally load 2 batches of files into a bronze table, confirming COPY INTO doesn't reprocess the first batch.

In [0]:
create table if not exists cyntexa_dev.bronze.sales_raw

In [0]:
copy into cyntexa_dev.bronze.sales_raw
from "/Volumes/cyntexa_dev/bronze/raw/sales/"
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')


In [0]:
select count(*) from cyntexa_dev.bronze.sales_raw

Number of rows inserted : 78

Total numbr of records after first run :  78

In [0]:
copy into cyntexa_dev.bronze.sales_raw
from "/Volumes/cyntexa_dev/bronze/raw/sales/"
fileformat = csv
format_options ('header' = 'true')
copy_options ('mergeSchema' = 'true')


In [0]:
select count(*) from cyntexa_dev.bronze.sales_raw

Number of rows inserted : 77

Total number of records after second run :  155

This confirms COPY INTO did not reload files form first batch 

In [0]:
-- 3. Query an old version of the table with both VERSION AS OF and TIMESTAMP AS OF.

In [0]:
desc history cyntexa_dev.bronze.products

In [0]:
select * from cyntexa_dev.bronze.products version as of 2

In [0]:
select * from cyntexa_dev.bronze.products timestamp as of '2026-08-26T08:29:43.000+00:00' --version 3

### 2. Intermediate Tasks

In [0]:
-- 4. Evolve the table's schema two ways: append a new column using mergeSchema, then change an existing column's type using overwriteSchema; document the difference in what each requires.

#### Merge schema

In [0]:
%python
df = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers1.csv', header = True, inferSchema = True)
df.display()

In [0]:
%python
df.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.sales_schema_evolution")

In [0]:
%python
df2 = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers2.csv', header = True, inferSchema = True)
df2.display()

In [0]:
%python
df2.write.mode("append").option('mergeSchema', True).saveAsTable("cyntexa_dev.sales.sales_schema_evolution")

In [0]:
select * from cyntexa_dev.sales.sales_schema_evolution

#### Overwrite schema

In [0]:
%python
df3 = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/customers/customers3.csv', header=True, inferSchema=True,sep='|')
df3.display()

In [0]:
%python
from pyspark.sql.functions import *
df3 = df3.withColumn("customer_id", concat(lit("C"), col("customer_id")))
df3.display()

In [0]:
%python
df3.printSchema()

Overwriting schmema by changing customer_id data type to string

In [0]:
%python
df3.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("cyntexa_dev.sales.sales_schema_overwrite")

In [0]:
select * from cyntexa_dev.sales.sales_schema_overwrite

In [0]:
-- 5. Set up an Autoloader stream ingesting from a folder, then drop 2 more files into the folder and confirm they're picked up automatically.

In [0]:
%python
from pyspark.sql import functions as F
df = spark.readStream.format('cloudFiles') \
    .option('cloudFiles.format','csv') \
    .option('cloudFiles.inferColumnTypes','true') \
    .option('cloudFiles.schemaLocation', "/Volumes/cyntexa_dev/bronze/schema/sales") \
    .option('header','true') \
    .load('/Volumes/cyntexa_dev/bronze/raw/sales/')

df = df.withColumn('ingestion_timestamp', F.current_timestamp())

df.writeStream.format('delta') \
    .outputMode('append') \
    .option('checkpointLocation','/Volumes/cyntexa_dev/bronze/checkpoint/sales') \
    .trigger(availableNow=True) \
    .toTable('cyntexa_dev.bronze.sales_stream')

In [0]:
select count(*) from cyntexa_dev.bronze.sales_stream

In [0]:
-- 6. Use RESTORE to roll a table back to a version before a bad schema change, and describe what happens to the versions that were created after the point you restored to.

In [0]:
desc history cyntexa_dev.bronze.products

In [0]:
restore table cyntexa_dev.bronze.products version as of 2 

In [0]:
desc history cyntexa_dev.bronze.products

The RESTORE did not delete the versions that were create after selected version, instead it ceated a new version containning the state of the selected version. The previous versions can still be queried using time travel

### 3. Advanced Tasks

7. Compare all four ingestion patterns covered (batch CTAS, COPY INTO, Autoloader, Lakeflow Declarative Pipelines) on cost, latency, and operational complexity, and recommend which one Cyntexa should use for a file source that arrives unpredictably throughout the day.


- Batch CTAS: It is most suitable for one time or schedules batch loads, it is simple and inexpensive but not suitable when files arrive unpredictably because data is only loaded when quary is executed

- COPY INTO: Suitable for incremental file ingstion and is relatively simple to operate. It checks for new files as they arive and can process them with low latency making it suitable unexpected files arrival.

- Auto Loader: designed for incremental as well as aontinuous file ingestion. It automatically detects new files as they arrive and can process them with low latency.

- Lakeflow Declarative Pipelines: provides a managed way to built and orchestrate data pipeline and can use Auto Loader for file ingestion. 

For Cyntexa's file source, where files arrive unpredictably throughout the day, Auto Loader is the preferred ingestion pattern.It is designed to incrementally detect and process newly arriving files without requiring a complete batch reload. 

In [0]:
-- 8. Design a recovery runbook: if a bad file corrupts the silver table at 2am, walk through the exact commands (DESCRIBE HISTORY, RESTORE or time travel + overwrite) an on-call engineer would run.

In [0]:
-- Check the table history
desc history cyntexa_dev.bronze.products

In [0]:
-- Identify last known good version, inspect that version before restoring
select * from cyntexa_dev.bronze.products version as of 4

In [0]:
select count(*) from cyntexa_dev.bronze.products version as of 4

In [0]:
-- After confirming the version is good, restore the table to that version
restore table cyntexa_dev.bronze.products version as of 4;
-- Confirm the table is restored
select count(*) from cyntexa_dev.bronze.products

In [0]:
-- Then inspect the latest history
desc history cyntexa_dev.bronze.products

In [0]:
-- 9. (Data Analyst) Using DESCRIBE HISTORY, produce a 'data freshness' report showing how frequently a given table is actually updated, to validate an SLA claim made to a business stakeholder.

In [0]:
-- get history table 
desc history cyntexa_dev.silver.products

In [0]:
select version,
operation,
timestamp,
lag(timestamp) over (order by timestamp) as prev_ts,
timestamp - lag(timestamp) over (order by timestamp) as difference
from (describe history cyntexa_dev.silver.products)